## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Import all necessary libraries for the tutorial
from bedrock_agentcore_starter_toolkit import Runtime 
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from pathlib import Path
from strands import Agent,tool 
from strands.models import BedrockModel  
from strands.tools.mcp.mcp_client import MCPClient 
from streamable_http_sigv4 import streamablehttp_client_with_sigv4  
from strands_tools import agent_graph, retrieve
import boto3  
import getpass 
import json
import logging  
import uuid 
import os


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [ ]:
session = boto3.Session()

credentials = session.get_credentials()

region = session.region_name

cf_client = boto3.client(
    "cloudformation", region_name=region
)

agentcore_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

identity_client = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
)

s3_client = session.client('s3')

sts_client = session.client('sts')
account_id = sts_client.get_caller_identity()["Account"]

In [ ]:
# Define configuration variables for the tutorial resources

# Gateway and target names
#gateway_name = "aws-host-support-knowledge-mcp-gatewat"  # Name for the AgentCore Gateway

# Agent configuration
agent_name = "AWS_Support_knowledge_QA_Agent"  # Name for the deployed agent in AgentCore Runtime

## Step 6: Deploy to Amazon Bedrock AgentCore Runtime

Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

### Step 6.1: Configure Amazon Bedrock AgentCore Runtime

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<center>

![prereq](./images/configure.png)

</center>

In [ ]:
# Initialize the AgentCore Runtime manager
# This object handles the deployment of our agent to AWS Lambda
agentcore_runtime = Runtime()

# Configure the runtime deployment settings
# This prepares all necessary AWS resources for deploying the agent
response = agentcore_runtime.configure(
    entrypoint="aws_support_agent.py",  
    auto_create_ecr=True, 
    requirements_file="requirements.txt",  
    region=region,  
    agent_name=agent_name,
    auto_create_execution_role=True,
    non_interactive=True
)

# Display the configuration response
response

### Step 6.2: Launch Amazon Bedrock AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<center>

![prereq](./images/launch.png)

</center>

In [ ]:
msg = "Launching Agent to AgentCore Runtime (This might take several minutes...)"
print(msg)
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)
print("✅ Launch completed")

In [ ]:
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

os.environ['AGENT_ARN'] = launch_result.agent_arn

### Step 6.3: Invoke AgentCore runtime

Finally, we can invoke our AgentCore Runtime with a payload

<center>

![prereq](./images/invoke.png)

</center>

In [ ]:
import time

print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
max_wait_time = 1800  # 30 minutes
wait_time = 0

while status not in end_status and wait_time < max_wait_time:
    msg = f"Status: {status} - waiting... ({wait_time}s/{max_wait_time}s)"
    print(msg)
    time.sleep(10)
    wait_time += 10

    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if wait_time >= max_wait_time:
    print(f"⚠️ Timeout reached. Current status: {status}")
elif status == 'READY':
    print(f"✅ AgentCore Runtime is READY!")
else:
    print(f"⚠️ AgentCore Runtime status: {status}")

print(f"Final status: {status}")

## 4.Grant premission

In [ ]:
grant "ssm:GetParameter"/"bedrock:Retrieve"/"bedrock-agentcore:InvokeGateway" to this agentcore runtime 

In [ ]:
from agent_client import test_invoke_agent

# Test the deployed support case agent\n
test_invoke_agent()